In [2]:
import ROOT
import numpy as np
import math


def root_minimize(func,
                ndim,
                minimizerName="Minuit2",
                algoName="",
                mg_init=None,
                eps_init=None,
                a1_init=None,
                a2_init=None,
                stepSize=None,
                maxFunctionCalls=1000000,
                maxIterations=10000,
                tolerance=1e-8,
                printLevel=1):
    """
    Generic ROOT Minimization Wrapper with Confidence Level Calculation.

    Parameters
    ----------
    func : callable
        Function to minimize. Should accept a list or numpy array of length `ndim`.

    ndim : int
        Number of parameters to minimize.

    minimizerName : str, default="Minuit2"
        Minimizer to use (Minuit, Minuit2, GSLMultiMin, GSLSimAn, Genetic, etc.).
    
    algoName : str, default=""
        Specific algorithm (Migrad, BFGS, ConjugateFR, Simplex, etc.).
    
    mg_init : float or None
        Initial guess for parameter 'mg'. Defaults to 0.0 if None.
    
    eps_init : float or None
        Initial guess for parameter 'eps'. Defaults to 0.0 if None.
    
    a1_init : float or None
        Initial guess for parameter 'a1'. Defaults to 0.0 if None.
    
    a2_init : float or None
        Initial guess for parameter 'a2'. Defaults to 0.0 if None.
    
    stepSize : list of floats or None
        Step sizes for each parameter. Defaults to 0.01 for all.
    
    maxFunctionCalls : int, default=1000000
        Maximum allowed function evaluations.
    
    maxIterations : int, default=10000
        Maximum allowed iterations.
    
    tolerance : float, default=1e-8
        Desired tolerance for convergence.
    
    printLevel : int, default=1
        Verbosity of the minimizer (0=quiet, 1=normal, 2=verbose).

    Returns
    -------
    dict
        Dictionary containing:
        - 'success': bool, whether minimization converged successfully
        - 'x': numpy array, parameter values at minimum
        - 'status': int, minimizer status (0 = success)
        - 'hesse_errors': numpy array, symmetric Hesse errors
        - 'minos_errors_low': numpy array, lower MINOS errors
        - 'minos_errors_up': numpy array, upper MINOS errors
    """

    #-------------------
    #  SET STARTING POINT
    #-------------------

    param_names = ["mg", "eps", "a1", "a2"]
    init_map = [mg_init, eps_init, a1_init, a2_init]
    startPoint = []
    for i in range(ndim):
        if init_map[i] is not None:
            startPoint.append(init_map[i])
        else:
            startPoint.append(0.0)  # fallback default
    # --------------------------------------------------------------

    #-------------------
    #  SET STEP SIZE
    #-------------------
    if stepSize is None:
        stepSize = [0.01] * ndim

    #-------------------
    #  SET CONFIDENCE LEVEL FOR 4D CASE 90% CL
    #-------------------
    errordef = 7.78

    # import scipy
    # import scipy.stats
    # print(scipy.stats.chi2.ppf(0.9 , df=4))

    # cl_to_errordef_4d = {
    #     68.3: 4.72,
    #     90.0: 7.78,
    #     95.0: 9.49,
    #     99.0: 13.28
    # }

    
    #-------------------
    #  CREATE MINIMIZER
    #-------------------

    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizerName}\"")
    

    #-------------------
    #  SET OPTIONS
    #-------------------

    minimizer.SetMaxFunctionCalls(maxFunctionCalls)
    minimizer.SetMaxIterations(maxIterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetPrintLevel(printLevel)
    minimizer.SetErrorDef(errordef)
    f = ROOT.Math.Functor(func, ndim)
    minimizer.SetFunction(f)

    
    variable = list(startPoint)

    #-------------------
    #  SET PARAMETERS
    #-------------------

    # renaming variable names to match model parameters and set parameters 
    for i in range(ndim):
        if i < len(param_names):
            name = param_names[i]
        else:
            name = f"x{i}"
        minimizer.SetVariable(i, name, variable[i], stepSize[i])

    #could replace the code above by the following code to set parameters without renaming

    # for i in range(ndim):
    #     minimizer.SetVariable(i, f"x{i}", variable[i], stepSize[i])


    #-------------------
    #  RUN MINIMIZATION
    #-------------------

    minimization = minimizer.Minimize()
    if not minimization:
        return {'success': False}
    

    #-------------------
    # GET HESSE ERROR
    #-------------------

    # Create empty arrays to store the results
    xs = np.zeros(ndim)           # parameter values at minimum
    hesse_errors = np.zeros(ndim) # symmetric Hesse errors

    # Loop over each parameter and extract the value and Hesse error
    for i in range(ndim):
        xs[i] = minimizer.X()[i]          # get the fitted value of parameter i
        hesse_errors[i] = minimizer.Errors()[i]  # get the Hesse error for parameter i


    #-------------------
    # GET MINOS ERROR
    #-------------------
    
    # Initialize arrays to store MINOS errors
    minos_errors_low = np.zeros(ndim)
    minos_errors_up = np.zeros(ndim)

    # Temporary arrays for ROOT's GetMinosError
    errLow = np.zeros(1, dtype=np.float64)
    errUp  = np.zeros(1, dtype=np.float64)

    for i in range(ndim):
        success = minimizer.GetMinosError(i, errLow, errUp)
        if success:
            minos_errors_low[i] = errLow[0]
            minos_errors_up[i] = errUp[0]
        else:
            # fallback to Hesse errors if MINOS fails
            minos_errors_low[i] = -hesse_errors[i]
            minos_errors_up[i] = hesse_errors[i]



    # print results
    print("\nMinimization results (values ± Hesse ± MINOS):")
    for i in range(ndim):
        print(f"{param_names[i]}: {xs[i]:.6f} "
              f"± {hesse_errors[i]:.6f} "
              f"[{minos_errors_low[i]:+.6f}, {minos_errors_up[i]:+.6f}]")

    print(f"\nStatus: {minimizer.Status()} (0 = success)\n")
    # ----------------------

    return {
        'success': minimization and minimizer.Status() == 0,
        'x': xs,
        'status': minimizer.Status(),
        'hesse_errors': hesse_errors,
        'minos_errors_low': minos_errors_low,
        'minos_errors_up': minos_errors_up,
    }


In [ ]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo COMPLEXA com 4 parâmetros livres:
    f(x) = mg * exp(-eps * x[0]) + a1 * x[0] + a2 * x[0]**2
    
    par[0] = mg  (magnitude/amplitude)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (coeficiente linear)
    par[3] = a2  (coeficiente quadrático)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]

    # Prevenir overflow no exp
    arg = eps * (x[0] - a1)
    if arg > 100:
        logistic = 0.0
    elif arg < -100:
        logistic = 1.0
    else:
        logistic = 1.0 / (1.0 + ROOT.TMath.Exp(arg))

    return mg * logistic + a2 * x[0]

def fit_data(func_model, x_data, y_data, y_errors, initial_params, param_limits, xmin, xmax):
    """
    Função para realizar o ajuste usando LeastSquareFit com ROOT
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Nomear os parâmetros
    param_names = ['mg', 'eps', 'a1', 'a2']
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais e limites
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
        if param_limits[i] is not None:
            func.SetParLimits(i, param_limits[i][0], param_limits[i][1])
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    # Configurações do minimizador Minuit2
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(0)
    fitter.Config().MinimizerOptions().SetStrategy(2)
    fitter.Config().MinimizerOptions().SetPrecision(1e-8)  
    fitter.Config().MinimizerOptions().SetTolerance(1e-3)
    fitter.Config().MinimizerOptions().SetMaxFunctionCalls(50000)
    fitter.Config().MinimizerOptions().SetMaxIterations(50000)
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados
    result = fitter.Result()
    
    output = {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'param_names': param_names,
        'valid': result.IsValid(),
        'status': result.Status()
    }
    
    return output

# =============================================
# EXEMPLO COM FUNÇÃO COMPLEXA - 4 PARÂMETROS
# =============================================


true_params = {'mg': 8.0, 'eps': 1.2, 'a1': 5.0, 'a2': 0.3}
np.random.seed(42)
x_data = np.linspace(0.5, 10.0, 30)

# Dados simulados da função estável
y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])

noise_level = 0.3
y_data = y_true + np.random.normal(0, noise_level, len(x_data))
y_errors = np.full(len(x_data), noise_level)

# Chutes iniciais razoáveis
initial_params = [5.0, 0.5, 4.0, 0.1]
param_limits = [
    (0.1, 50.0),     # mg
    (0.01, 10.0),    # eps
    (0.1, 10.0),     # a1
    (-2.0, 2.0)      # a2
]

# Executar o ajuste
result = fit_data(model_function, x_data, y_data, y_errors, initial_params, 
                  param_limits, x_data.min(), x_data.max())

# =============================================
# APRESENTAÇÃO SIMPLIFICADA DOS RESULTADOS
# =============================================

print("=" * 60)
print("RESULTADOS DO AJUSTE")
print("=" * 60)

if result['valid']:
    print(f"\nChi²/DOF = {result['chi2_dof']:.4f}\n")
    
    print("PARÂMETROS AJUSTADOS:")
    for i, name in enumerate(result['param_names']):
        param_val = result['parameters'][i]
        param_err = result['errors'][i]
        print(f"  {name:4s} = {param_val:8.4f} ± {param_err:.4f}")
else:
    print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
    print("\nParâmetros tentados:")
    for i, name in enumerate(result['param_names']):
        print(f"  {name:4s} = {result['parameters'][i]:8.4f}")
    
print("=" * 60)

In [ ]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo COMPLEXA com 4 parâmetros livres:
    f(x) = mg * exp(-eps * x[0]) + a1 * x[0] + a2 * x[0]**2
    
    par[0] = mg  (magnitude/amplitude)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (coeficiente linear)
    par[3] = a2  (coeficiente quadrático)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]

    # Prevenir overflow no exp
    arg = eps * (x[0] - a1)
    if arg > 100:
        logistic = 0.0
    elif arg < -100:
        logistic = 1.0
    else:
        logistic = 1.0 / (1.0 + ROOT.TMath.Exp(arg))

    return mg * logistic + a2 * x[0]

def fit_data(func_model, x_data, y_data, y_errors, initial_params, param_limits, xmin, xmax):
    """
    Função para realizar o ajuste usando o método Fit() direto do TGraphErrors
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Nomear os parâmetros
    param_names = ['mg', 'eps', 'a1', 'a2']
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais e limites
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
        if param_limits[i] is not None:
            func.SetParLimits(i, param_limits[i][0], param_limits[i][1])
    
    # Fazer o fit diretamente
    # Opções: "S" = retorna TFitResultPtr, "Q" = quiet, "R" = usa range da função
    fit_result = graph.Fit(func, "SQR")
    
    # Extrair resultados
    output = {
        'chi2': fit_result.Chi2(),
        'ndf': fit_result.Ndf(),
        'chi2_dof': fit_result.Chi2() / fit_result.Ndf() if fit_result.Ndf() > 0 else 0.0,
        'parameters': [fit_result.Parameter(i) for i in range(fit_result.NPar())],
        'errors': [fit_result.ParError(i) for i in range(fit_result.NPar())],
        'param_names': param_names,
        'valid': fit_result.IsValid(),
        'status': fit_result.Status()
    }
    
    return output

# =============================================
# EXEMPLO COM FUNÇÃO COMPLEXA - 4 PARÂMETROS
# =============================================

true_params = {'mg': 8.0, 'eps': 1.2, 'a1': 5.0, 'a2': 0.3}
np.random.seed(42)
x_data = np.linspace(0.5, 10.0, 30)

# Dados simulados da função estável
y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])

noise_level = 0.3
y_data = y_true + np.random.normal(0, noise_level, len(x_data))
y_errors = np.full(len(x_data), noise_level)

# Chutes iniciais razoáveis
initial_params = [5.0, 0.5, 4.0, 0.1]
param_limits = [
    (0.1, 50.0),     # mg
    (0.01, 10.0),    # eps
    (0.1, 10.0),     # a1
    (-2.0, 2.0)      # a2
]

# Executar o ajuste
result = fit_data(model_function, x_data, y_data, y_errors, initial_params, 
                  param_limits, x_data.min(), x_data.max())

# =============================================
# APRESENTAÇÃO SIMPLIFICADA DOS RESULTADOS
# =============================================

print("=" * 60)
print("RESULTADOS DO AJUSTE")
print("=" * 60)

if result['valid']:
    print(f"\nChi²/DOF = {result['chi2_dof']:.4f}\n")
    
    print("PARÂMETROS AJUSTADOS:")
    for i, name in enumerate(result['param_names']):
        param_val = result['parameters'][i]
        param_err = result['errors'][i]
        print(f"  {name:4s} = {param_val:8.4f} ± {param_err:.4f}")
else:
    print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
    print("\nParâmetros tentados:")
    for i, name in enumerate(result['param_names']):
        print(f"  {name:4s} = {result['parameters'][i]:8.4f}")
    
print("=" * 60)

In [ ]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo COMPLEXA com 4 parâmetros livres:
    f(x) = mg * exp(-eps * x[0]) + a1 * x[0] + a2 * x[0]**2
    
    par[0] = mg  (magnitude/amplitude)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (coeficiente linear)
    par[3] = a2  (coeficiente quadrático)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]

    # Prevenir overflow no exp
    arg = eps * (x[0] - a1)
    if arg > 100:
        logistic = 0.0
    elif arg < -100:
        logistic = 1.0
    else:
        logistic = 1.0 / (1.0 + ROOT.TMath.Exp(arg))

    return mg * logistic + a2 * x[0]


def fit_data(func_model, x_data, y_data, y_errors, initial_params, param_limits, 
             xmin, xmax, minimizerName="Minuit2", algoName="Migrad", **fit_options):
    """
    Função para realizar o ajuste usando graph.Fit() com configuração do minimizador
    
    Parameters
    ----------
    func_model : callable
        Modelo a ser ajustado
    x_data : array-like
        Dados de x
    y_data : array-like
        Dados de y
    y_errors : array-like
        Erros em y
    initial_params : list
        Parâmetros iniciais [mg, eps, a1, a2]
    param_limits : list of tuples
        Limites para cada parâmetro [(min, max), ...]
    xmin : float
        Limite inferior do range de ajuste
    xmax : float
        Limite superior do range de ajuste
    minimizerName : str, default="Minuit2"
        Nome do minimizador (Minuit2, Minuit, GSLMultiMin, etc.)
    algoName : str, default="Migrad"
        Algoritmo específico (Migrad, Simplex, Minimize, etc.)
    **fit_options : dict
        Opções adicionais:
        - tolerance: float, tolerância (default: 1e-8)
        - maxIterations: int, máximo de iterações (default: 10000)
        - maxFunctionCalls: int, máximo de chamadas (default: 1000000)
        - printLevel: int, nível de verbosidade (default: 1)
        - errordef: float, definição de erro para CL (default: 7.78 para 90% CL 4D)
        - strategy: int, estratégia do Minuit (0=rápido, 1=normal, 2=seguro)
    
    Returns
    -------
    dict
        Dicionário com resultados do ajuste
    """
    n_points = len(x_data)
    param_names = ['mg', 'eps', 'a1', 'a2']
    npar = len(initial_params)
    
    # Criar TGraphErrors
    graph = ROOT.TGraphErrors(n_points)
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Criar a função TF1
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Nomear os parâmetros
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais e limites
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
        if param_limits[i] is not None:
            func.SetParLimits(i, param_limits[i][0], param_limits[i][1])
    
    # Configurar o minimizador através do ROOT.Math.MinimizerOptions
    ROOT.Math.MinimizerOptions.SetDefaultMinimizer(minimizerName, algoName)
    
    # Configurar opções adicionais
    if 'tolerance' in fit_options:
        ROOT.Math.MinimizerOptions.SetDefaultTolerance(fit_options['tolerance'])
    else:
        ROOT.Math.MinimizerOptions.SetDefaultTolerance(1e-8)
    
    if 'maxIterations' in fit_options:
        ROOT.Math.MinimizerOptions.SetDefaultMaxIterations(fit_options['maxIterations'])
    else:
        ROOT.Math.MinimizerOptions.SetDefaultMaxIterations(10000)
    
    if 'maxFunctionCalls' in fit_options:
        ROOT.Math.MinimizerOptions.SetDefaultMaxFunctionCalls(fit_options['maxFunctionCalls'])
    else:
        ROOT.Math.MinimizerOptions.SetDefaultMaxFunctionCalls(1000000)
    
    if 'printLevel' in fit_options:
        ROOT.Math.MinimizerOptions.SetDefaultPrintLevel(fit_options['printLevel'])
    else:
        ROOT.Math.MinimizerOptions.SetDefaultPrintLevel(1)
    
    if 'errordef' in fit_options:
        ROOT.Math.MinimizerOptions.SetDefaultErrorDef(fit_options['errordef'])
    else:
        ROOT.Math.MinimizerOptions.SetDefaultErrorDef(7.78)  # 90% CL for 4D
    
    if 'strategy' in fit_options:
        ROOT.Math.MinimizerOptions.SetDefaultStrategy(fit_options['strategy'])
    
    # Fazer o fit usando o minimizador configurado
    # Opções: "S" = retorna TFitResultPtr, "Q" = quiet, "R" = usa range da função
    fit_result = graph.Fit(func, "SQR")
    
    # Extrair resultados
    output = {
        'chi2': fit_result.Chi2(),
        'ndf': fit_result.Ndf(),
        'chi2_dof': fit_result.Chi2() / fit_result.Ndf() if fit_result.Ndf() > 0 else 0.0,
        'parameters': [fit_result.Parameter(i) for i in range(fit_result.NPar())],
        'errors': [fit_result.ParError(i) for i in range(fit_result.NPar())],
        'param_names': param_names,
        'valid': fit_result.IsValid(),
        'status': fit_result.Status(),
        'function': func,
        'fit_result': fit_result  # Retornar TFitResultPtr completo para acesso a mais info
    }
    
    # Calcular MINOS errors se disponível
    if fit_result.IsValid():
        minos_errors_low = []
        minos_errors_up = []
        
        for i in range(npar):
            try:
                err_low = fit_result.LowerError(i)
                err_up = fit_result.UpperError(i)
                minos_errors_low.append(err_low)
                minos_errors_up.append(err_up)
            except:
                # Fallback para erros Hesse se MINOS não disponível
                minos_errors_low.append(-output['errors'][i])
                minos_errors_up.append(output['errors'][i])
        
        output['minos_errors_low'] = minos_errors_low
        output['minos_errors_up'] = minos_errors_up
    
    return output


# =============================================
# EXEMPLO DE USO
# =============================================

if __name__ == "__main__":
    # Dados simulados
    true_params = {'mg': 8.0, 'eps': 1.2, 'a1': 5.0, 'a2': 0.3}
    np.random.seed(42)
    x_data = np.linspace(0.5, 10.0, 30)
    
    y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])
    
    noise_level = 0.3
    y_data = y_true + np.random.normal(0, noise_level, len(x_data))
    y_errors = np.full(len(x_data), noise_level)
    
    # Chutes iniciais
    initial_params = [5.0, 0.5, 4.0, 0.1]
    param_limits = [
        (0.1, 50.0),     # mg
        (0.01, 10.0),    # eps
        (0.1, 10.0),     # a1
        (-2.0, 2.0)      # a2
    ]
    
    # ===== USAR graph.Fit() com Minuit2 configurado =====
    print("\n" + "="*60)
    print("AJUSTE COM graph.Fit() usando Minuit2")
    print("="*60)
    
    result = fit_data(
        model_function, x_data, y_data, y_errors,
        initial_params, param_limits,
        x_data.min(), x_data.max(),
        minimizerName="Minuit2",
        algoName="Migrad",
        tolerance=1e-6,
        maxIterations=10000,
        printLevel=1,
        errordef=7.78,  # 90% CL para 4D
        strategy=1
    )
    
    if result['valid']:
        print(f"\nChi²/DOF = {result['chi2_dof']:.4f}\n")
        print("PARÂMETROS AJUSTADOS:")
        for i, name in enumerate(result['param_names']):
            print(f"  {name:4s} = {result['parameters'][i]:8.4f} ± {result['errors'][i]:.4f}")
        
        # Se MINOS errors disponíveis
        if 'minos_errors_low' in result:
            print("\nCom MINOS errors:")
            for i, name in enumerate(result['param_names']):
                print(f"  {name:4s} = {result['parameters'][i]:8.4f} "
                      f"[{result['minos_errors_low'][i]:+.4f}, "
                      f"{result['minos_errors_up'][i]:+.4f}]")
    else:
        print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
    
    print("\n" + "="*60)

In [ ]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo COMPLEXA com 4 parâmetros livres:
    f(x) = mg * exp(-eps * x[0]) + a1 * x[0] + a2 * x[0]**2
    
    par[0] = mg  (magnitude/amplitude)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (coeficiente linear)
    par[3] = a2  (coeficiente quadrático)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]

    # Prevenir overflow no exp
    arg = eps * (x[0] - a1)
    if arg > 100:
        logistic = 0.0
    elif arg < -100:
        logistic = 1.0
    else:
        logistic = 1.0 / (1.0 + ROOT.TMath.Exp(arg))

    return mg * logistic + a2 * x[0]


def fit_data(func_model, x_data, y_data, y_errors, initial_params, param_limits, 
             xmin, xmax, root_minimize, minimizerName="Minuit2", algoName="Migrad", 
             **minimize_options):
    """
    Função para realizar o ajuste usando root_minimize com chi-square built-in do ROOT
    
    Parameters
    ----------
    func_model : callable
        Modelo a ser ajustado
    x_data : array-like
        Dados de x
    y_data : array-like
        Dados de y
    y_errors : array-like
        Erros em y
    initial_params : list
        Parâmetros iniciais [mg, eps, a1, a2]
    param_limits : list of tuples
        Limites para cada parâmetro [(min, max), ...]
    xmin : float
        Limite inferior do range de ajuste
    xmax : float
        Limite superior do range de ajuste
    root_minimize : callable
        Função root_minimize para realizar a minimização
    minimizerName : str, default="Minuit2"
        Nome do minimizador (Minuit2, Minuit, GSLMultiMin, etc.)
    algoName : str, default="Migrad"
        Algoritmo específico (Migrad, Simplex, Minimize, etc.)
    **minimize_options : dict
        Opções adicionais para root_minimize:
        - stepSize: list, tamanhos de passo para cada parâmetro
        - tolerance: float, tolerância (default: 1e-8)
        - maxIterations: int, máximo de iterações (default: 10000)
        - maxFunctionCalls: int, máximo de chamadas (default: 1000000)
        - printLevel: int, nível de verbosidade (default: 1)
    
    Returns
    -------
    dict
        Dicionário com resultados do ajuste incluindo MINOS errors
    """
    n_points = len(x_data)
    param_names = ['mg', 'eps', 'a1', 'a2']
    npar = len(initial_params)
    
    # Criar TGraphErrors (necessário para chi-square built-in)
    graph = ROOT.TGraphErrors(n_points)
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Criar a função TF1
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Nomear os parâmetros
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
    
    # Criar função chi-square usando o método built-in do ROOT
    def chi2_func(params):
        # Atualizar parâmetros da TF1
        for i in range(npar):
            func.SetParameter(i, params[i])
        # Calcular chi2 usando método built-in
        return graph.Chisquare(func)
    
    # Preparar argumentos para root_minimize
    minimize_args = {
        'func': chi2_func,
        'ndim': npar,
        'minimizerName': minimizerName,
        'algoName': algoName,
        'mg_init': initial_params[0],
        'eps_init': initial_params[1],
        'a1_init': initial_params[2],
        'a2_init': initial_params[3],
    }
    
    # Adicionar opções do usuário
    minimize_args.update(minimize_options)
    
    # Executar minimização usando root_minimize
    result = root_minimize(**minimize_args)
    
    if result['success']:
        # Atualizar função com parâmetros otimizados
        for i in range(npar):
            func.SetParameter(i, result['x'][i])
        
        # Calcular chi2 final usando built-in
        chi2 = graph.Chisquare(func)
        ndf = n_points - npar
        
        output = {
            'chi2': chi2,
            'ndf': ndf,
            'chi2_dof': chi2 / ndf if ndf > 0 else 0.0,
            'parameters': result['x'].tolist(),
            'errors': result['hesse_errors'].tolist(),
            'minos_errors_low': result['minos_errors_low'].tolist(),
            'minos_errors_up': result['minos_errors_up'].tolist(),
            'param_names': param_names,
            'valid': True,
            'status': result['status'],
            'function': func
        }
    else:
        output = {
            'chi2': 0.0,
            'ndf': 0,
            'chi2_dof': 0.0,
            'parameters': initial_params,
            'errors': [0.0] * npar,
            'minos_errors_low': [0.0] * npar,
            'minos_errors_up': [0.0] * npar,
            'param_names': param_names,
            'valid': False,
            'status': -1,
            'function': func
        }
    
    return output


# =============================================
# EXEMPLO DE USO
# =============================================

if __name__ == "__main__":
    # Importar root_minimize
    # from root_minimize_module import root_minimize  # ajuste o import conforme necessário
    
    # Dados simulados
    true_params = {'mg': 8.0, 'eps': 1.2, 'a1': 5.0, 'a2': 0.3}
    np.random.seed(42)
    x_data = np.linspace(0.5, 10.0, 30)
    
    y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])
    
    noise_level = 0.3
    y_data = y_true + np.random.normal(0, noise_level, len(x_data))
    y_errors = np.full(len(x_data), noise_level)
    
    # Chutes iniciais
    initial_params = [5.0, 0.5, 4.0, 0.1]
    param_limits = [
        (0.1, 50.0),     # mg
        (0.01, 10.0),    # eps
        (0.1, 10.0),     # a1
        (-2.0, 2.0)      # a2
    ]
    
    print("\n" + "="*60)
    print("AJUSTE USANDO root_minimize COM CHI-SQUARE BUILT-IN")
    print("="*60)
    
    # Exemplo de como usar (descomente quando tiver root_minimize importado)
    
    result = fit_data(
        model_function, x_data, y_data, y_errors,
        initial_params, param_limits,
        x_data.min(), x_data.max(),
        root_minimize=root_minimize,  # passar a função root_minimize
        minimizerName="Minuit2",
        algoName="Migrad",
        stepSize=[0.1, 0.01, 0.1, 0.01],
        tolerance=1e-6,
        maxIterations=10000,
        printLevel=1
    )
    
    if result['valid']:
        print(f"\nChi²/DOF = {result['chi2_dof']:.4f}")
        print(f"Chi² = {result['chi2']:.4f}")
        print(f"NDF = {result['ndf']}\n")
        
        print("PARÂMETROS AJUSTADOS (com MINOS):")
        for i, name in enumerate(result['param_names']):
            print(f"  {name:4s} = {result['parameters'][i]:8.4f} "
                  f"± {result['errors'][i]:.4f} "
                  f"[{result['minos_errors_low'][i]:+.4f}, "
                  f"{result['minos_errors_up'][i]:+.4f}]")
    else:
        print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
    
    
    print("\nNota: Descomente o código acima e importe root_minimize para executar")
    print("="*60)

In [7]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo COMPLEXA e NÃO-TRIVIAL com 4 parâmetros:
    f(x) = mg * exp(-eps * x[0]) * cos(a1 * x[0]) + a2 * x[0]
    
    Esta função combina:
    - Decaimento exponencial amortecido
    - Oscilação cossenoidal
    - Tendência linear
    
    Parâmetros:
    par[0] = mg  (magnitude/amplitude do decaimento oscilante)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (frequência angular da oscilação)
    par[3] = a2  (coeficiente linear da tendência)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]
    
    # Prevenir overflow no exp
    exp_arg = -eps * x[0]
    if exp_arg < -100:
        exp_term = 0.0
    elif exp_arg > 100:
        exp_term = 1e43  # valor muito grande
    else:
        exp_term = ROOT.TMath.Exp(exp_arg)
    
    # Componente oscilatória amortecida
    oscillatory = mg * exp_term * ROOT.TMath.Cos(a1 * x[0])
    
    # Componente linear
    linear = a2 * x[0]
    
    return oscillatory + linear


def fit_data(func_model, x_data, y_data, y_errors, initial_params, param_limits, 
             xmin, xmax, root_minimize, minimizerName="Minuit2", algoName="Migrad", 
             **minimize_options):
    """
    Função para realizar o ajuste usando root_minimize com chi-square built-in do ROOT
    
    Parameters
    ----------
    func_model : callable
        Modelo a ser ajustado
    x_data : array-like
        Dados de x
    y_data : array-like
        Dados de y
    y_errors : array-like
        Erros em y
    initial_params : list
        Parâmetros iniciais [mg, eps, a1, a2]
    param_limits : list of tuples
        Limites para cada parâmetro [(min, max), ...]
    xmin : float
        Limite inferior do range de ajuste
    xmax : float
        Limite superior do range de ajuste
    root_minimize : callable
        Função root_minimize para realizar a minimização
    minimizerName : str, default="Minuit2"
        Nome do minimizador (Minuit2, Minuit, GSLMultiMin, etc.)
    algoName : str, default="Migrad"
        Algoritmo específico (Migrad, Simplex, Minimize, etc.)
    **minimize_options : dict
        Opções adicionais para root_minimize
    
    Returns
    -------
    dict
        Dicionário com resultados do ajuste incluindo MINOS errors
    """
    n_points = len(x_data)
    param_names = ['mg', 'eps', 'a1', 'a2']
    npar = len(initial_params)
    
    # Criar TGraphErrors (necessário para chi-square built-in)
    graph = ROOT.TGraphErrors(n_points)
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Criar a função TF1
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Nomear os parâmetros
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais e limites
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
        if param_limits and i < len(param_limits):
            func.SetParLimits(i, param_limits[i][0], param_limits[i][1])
    
    # Criar função chi-square usando o método built-in do ROOT
    def chi2_func(params):
        # Atualizar parâmetros da TF1
        for i in range(npar):
            func.SetParameter(i, params[i])
        # Calcular chi2 usando método built-in
        return graph.Chisquare(func)
    
    # Preparar argumentos para root_minimize
    minimize_args = {
        'func': chi2_func,
        'ndim': npar,
        'minimizerName': minimizerName,
        'algoName': algoName,
        'mg_init': initial_params[0],
        'eps_init': initial_params[1],
        'a1_init': initial_params[2],
        'a2_init': initial_params[3],
    }
    
    # Adicionar limites se fornecidos
    # if param_limits:
    #     minimize_args['mg_limits'] = param_limits[0]
    #     minimize_args['eps_limits'] = param_limits[1]
    #     minimize_args['a1_limits'] = param_limits[2]
    #     minimize_args['a2_limits'] = param_limits[3]
    
    # Adicionar opções do usuário
    minimize_args.update(minimize_options)
    
    # Executar minimização usando root_minimize
    result = root_minimize(**minimize_args)
    
    if result['success']:
        # Atualizar função com parâmetros otimizados
        for i in range(npar):
            func.SetParameter(i, result['x'][i])
        
        # Calcular chi2 final usando built-in
        chi2 = graph.Chisquare(func)
        ndf = n_points - npar
        
        output = {
            'chi2': chi2,
            'ndf': ndf,
            'chi2_dof': chi2 / ndf if ndf > 0 else 0.0,
            'parameters': result['x'].tolist(),
            'errors': result['hesse_errors'].tolist(),
            'minos_errors_low': result['minos_errors_low'].tolist(),
            'minos_errors_up': result['minos_errors_up'].tolist(),
            'param_names': param_names,
            'valid': True,
            'status': result['status'],
            'function': func,
            'covariance_matrix': result.get('covariance_matrix', None)
        }
    else:
        output = {
            'chi2': 0.0,
            'ndf': 0,
            'chi2_dof': 0.0,
            'parameters': initial_params,
            'errors': [0.0] * npar,
            'minos_errors_low': [0.0] * npar,
            'minos_errors_up': [0.0] * npar,
            'param_names': param_names,
            'valid': False,
            'status': -1,
            'function': func,
            'covariance_matrix': None
        }
    
    return output


# =============================================
# EXEMPLO DE USO
# =============================================

if __name__ == "__main__":
    # Importar root_minimize
    # from root_minimize_module import root_minimize  # ajuste o import conforme necessário
    
    print("\n" + "="*70)
    print("AJUSTE DE FUNÇÃO COMPLEXA: OSCILAÇÃO AMORTECIDA + TENDÊNCIA LINEAR")
    print("="*70)
    
    # Parâmetros verdadeiros do modelo físico
    true_params = {
        'mg': 8.5,      # Magnitude da oscilação
        'eps': 0.25,    # Taxa de decaimento (amortecimento)
        'a1': 3.0,      # Frequência angular (rad/unidade)
        'a2': 0.6       # Inclinação da tendência linear
    }
    
    print("\nPARÂMETROS VERDADEIROS (usados para gerar dados):")
    for name, value in true_params.items():
        print(f"  {name:4s} = {value:8.4f}")
    
    # Gerar dados sintéticos
    np.random.seed(456)
    n_points = 45
    x_data = np.linspace(0.5, 12.0, n_points)
    
    # Calcular valores verdadeiros
    y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])
    
    # Adicionar ruído realista
    noise_level = 0.4
    y_data = y_true + np.random.normal(0, noise_level, len(x_data))
    y_errors = np.full(len(x_data), noise_level)
    
    # Chutes iniciais (propositalmente distantes dos valores verdadeiros)
    initial_params = [6.0, 0.15, 2.5, 0.4]
    
    # Limites físicos para os parâmetros
    param_limits = [
        (0.1, 30.0),     # mg: magnitude positiva
        (0.01, 2.0),     # eps: decaimento positivo
        (0.5, 10.0),     # a1: frequência positiva
        (-2.0, 3.0)      # a2: tendência linear
    ]
    
    print("\nPARÂMETROS INICIAIS (chute):")
    param_names = ['mg', 'eps', 'a1', 'a2']
    for i, name in enumerate(param_names):
        print(f"  {name:4s} = {initial_params[i]:8.4f}")
    
    print("\n" + "-"*70)
    print("Executando ajuste com Minuit2/Migrad...")
    print("-"*70)
    
    # Exemplo de uso (descomente quando tiver root_minimize importado)

    result = fit_data(
        model_function, x_data, y_data, y_errors,
        initial_params, param_limits,
        x_data.min(), x_data.max(),
        root_minimize=root_minimize,  # passar a função root_minimize
        minimizerName="Minuit2",
        algoName="Migrad",
        stepSize=[0.3, 0.01, 0.1, 0.05],
        tolerance=1e-8,
        maxIterations=50000,
        printLevel=0
    )
    
    if result['valid']:
        print("\n" + "="*70)
        print("RESULTADOS DO AJUSTE")
        print("="*70)
        print(f"\nQualidade do ajuste:")
        print(f"  Chi² / DOF = {result['chi2']:.4f} / {result['ndf']} = {result['chi2_dof']:.4f}")
        
        # Avaliar qualidade
        if result['chi2_dof'] < 1.5:
            quality = "EXCELENTE ✓"
        elif result['chi2_dof'] < 3.0:
            quality = "BOA"
        else:
            quality = "RUIM - revisar modelo"
        print(f"  Qualidade: {quality}")
        
        print("\n" + "-"*70)
        print("PARÂMETROS AJUSTADOS:")
        print("-"*70)
        print(f"{'Nome':4s} | {'Verdadeiro':>10s} | {'Ajustado':>10s} | {'Erro (σ)':>10s} | {'Desvio':>8s}")
        print("-"*70)
        
        for i, name in enumerate(result['param_names']):
            true_val = list(true_params.values())[i]
            fit_val = result['parameters'][i]
            error = result['errors'][i]
            deviation = abs(fit_val - true_val) / error if error > 0 else 0
            
            print(f"{name:4s} | {true_val:10.4f} | {fit_val:10.4f} | {error:10.4f} | "
                  f"{deviation:7.2f}σ")
        
        print("\n" + "-"*70)
        print("ERROS MINOS (assimétricos):")
        print("-"*70)
        for i, name in enumerate(result['param_names']):
            print(f"  {name:4s} = {result['parameters'][i]:8.4f} "
                  f"[{result['minos_errors_low'][i]:+.4f}, "
                  f"{result['minos_errors_up'][i]:+.4f}]")
        
    else:
        print("\n" + "="*70)
        print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
        print("="*70)
        print("\nPossíveis causas:")
        print("  - Parâmetros iniciais muito distantes dos verdadeiros")
        print("  - Limites muito restritivos")
        print("  - Função muito complexa para os dados")
        print("  - Máximo de iterações atingido")



AJUSTE DE FUNÇÃO COMPLEXA: OSCILAÇÃO AMORTECIDA + TENDÊNCIA LINEAR

PARÂMETROS VERDADEIROS (usados para gerar dados):
  mg   =   8.5000
  eps  =   0.2500
  a1   =   3.0000
  a2   =   0.6000

PARÂMETROS INICIAIS (chute):
  mg   =   6.0000
  eps  =   0.1500
  a1   =   2.5000
  a2   =   0.4000

----------------------------------------------------------------------
Executando ajuste com Minuit2/Migrad...
----------------------------------------------------------------------

Minimization results (values ± Hesse ± MINOS):
mg: 8.796894 ± 1.083946 [-1.041372, +1.132066]
eps: 0.254849 ± 0.039474 [-0.037310, +0.042098]
a1: 2.995193 ± 0.024514 [-0.024607, +0.024574]
a2: 0.603012 ± 0.023404 [-0.023404, +0.023404]

Status: 0 (0 = success)


RESULTADOS DO AJUSTE

Qualidade do ajuste:
  Chi² / DOF = 37.4102 / 41 = 0.9124
  Qualidade: EXCELENTE ✓

----------------------------------------------------------------------
PARÂMETROS AJUSTADOS:
-------------------------------------------------------------